# pyDBSI — Protocol calibration: one set of λ for every acquisition of a protocol

**Why.** Left free, every acquisition calibrates its own regularisation ($\lambda_\text{aniso}$, $\lambda_\text{iso}$)
and the choice follows that acquisition's noise: on 40 acquisitions of one protocol the selected index
correlated with the SNR (Spearman −0.75 for $\lambda_\text{aniso}$, −0.72 for $\lambda_\text{iso}$). In a group
comparison the pipeline would then manufacture differences between noisier and cleaner scans, and a
single grid step of $\lambda_\text{aniso}$ moved the median fiber fraction by 40 % between two sessions of the
same subject. Calibrate **once per protocol**, then fit every acquisition with the same numbers.

**What this notebook does.**
1. finds the acquisitions and their files, with *your* folder layout and file names (section 0);
2. checks every input without fitting: files, grids, orientation of mask and noise map, and that all
   acquisitions share the same **protocol fingerprint** (volumes, b=0 count, shells);
3. calibrates each acquisition (`DBSI_Adaptive().calibrate()`: the hyperparameter selection of the fit,
   no voxel fit) and saves its record, so an interrupted run resumes where it stopped;
4. aggregates the records into one calibration (`calibrate_protocol`, rule `'geomean'`: the λ that minimises
   the mean of the log GCV curves, so a noisier acquisition does not outvote a cleaner one), with bootstrap
   stability and each acquisition's distance from its own optimum;
5. writes the calibration JSON and prints **the command line** to fit any acquisition of the protocol with
   those numbers.

**The sample.** At least 2 acquisitions with the same protocol; 10–40 is a sensible range. If the study has
groups (patients / controls, sites, scanners), take a **balanced** sample: the calibration should not
belong to one group. Every acquisition needs the **thermal noise map** (see the README: the `dwidenoise
-noise` map divided by the N4 bias field): records without it are refused.

**Time.** Only the hyperparameter selection runs, no voxel fit: on a laptop about 3 minutes per acquisition
with 300 calibration voxels, roughly proportionally longer with the 1000-voxel default.

## 0. Configuration

In [ ]:
import os, glob, json, time
import numpy as np, pandas as pd, nibabel as nib
import matplotlib.pyplot as plt
from IPython.display import display

import dbsi_toolbox
from dbsi_toolbox import (DBSI_Adaptive, load_data, protocol_fingerprint, fingerprint_mismatches,
                          calibrate_protocol, save_protocol_calibration)

# ═════════════════════════════════════════════════════════════════════════════
#  WHERE THE ACQUISITIONS ARE  (edit this cell only)
# ═════════════════════════════════════════════════════════════════════════════
PROTOCOL_NAME = 'P3'                                    # goes into the JSON and every run report
OUTPUT_DIR    = '/path/to/calibration/P3'               # JSON, records, figures

# One directory per acquisition: a glob pattern ...
ACQUISITION_GLOB = '/path/to/study/sub-*/ses-*/dwi'
# ... or an explicit list (takes precedence when not empty)
ACQUISITION_DIRS = []

# The files of an acquisition, as glob patterns RELATIVE to its directory: each must match exactly one file.
FILES = dict(
    dwi   = '*_preprocessed_N4.nii.gz',      # preprocessed DWI (topup, eddy, N4)
    bval  = '*_corrected.bval',
    bvec  = '*_corrected.bvec',              # rotated by eddy
    mask  = '*_brain_mask.nii.gz',
    noise = '*_noise_N4.nii.gz',             # thermal noise map, same grid and scale as the DWI
)

# How to name an acquisition in the tables: its path relative to this root (None = the directory name)
ID_RELATIVE_TO = '/path/to/study'

# Optional: group of each acquisition id, only to print how balanced the sample is
GROUPS = {}                                   # e.g. {'sub-01/ses-01/dwi': 'patient', ...}

N_CALIBRATION_VOXELS = 1000                   # the fit default; more voxels = slower, a bit more stable
RULE                 = 'geomean'              # or 'median_index' (the other one is always reported too)
LAMBDA_ISO_CAP_RULE  = 'min'                  # 'min' (no acquisition over-regularised), 'median', 'none'
FORCE                = False                  # True = recalibrate acquisitions already done

## 1. Find the acquisitions and their files

Each pattern of `FILES` must match **exactly one** file in every acquisition directory: zero or several
matches stop here, with the names, so the patterns can be tightened. Nothing is loaded yet.

In [ ]:
dirs = list(ACQUISITION_DIRS) or sorted(d for d in glob.glob(ACQUISITION_GLOB) if os.path.isdir(d))
assert dirs, f'no acquisition directory found ({ACQUISITION_GLOB if not ACQUISITION_DIRS else "ACQUISITION_DIRS"})'

def acq_id(d):
    return os.path.relpath(d, ID_RELATIVE_TO) if ID_RELATIVE_TO else os.path.basename(os.path.normpath(d))

ACQ, problems = {}, []
for d in dirs:
    files = {}
    for k, pat in FILES.items():
        hits = sorted(glob.glob(os.path.join(d, pat)))
        if len(hits) != 1:
            problems.append(f'{acq_id(d)}: {k} = {pat!r} matches {len(hits)} files {[os.path.basename(h) for h in hits]}')
        else:
            files[k] = hits[0]
    if len(files) == len(FILES):
        ACQ[acq_id(d)] = files
print(f'{len(dirs)} directories, {len(ACQ)} with all their files')
for p in problems:
    print('  [!]', p)
assert not problems, 'tighten the patterns in FILES (or remove those acquisitions) and rerun'
display(pd.DataFrame({i: {k: os.path.basename(v) for k, v in f.items()} for i, f in ACQ.items()}).T)
if GROUPS:
    print('acquisitions per group:', pd.Series([GROUPS.get(i, '(none)') for i in ACQ]).value_counts().to_dict())

## 2. Check every input, without fitting

Per acquisition: volumes = b-values = b-vectors; mask and noise map on the DWI grid **and orientation**;
how much of the mask has a valid noise value; thermal SNR ($S_0/\sigma$ on the b=0 volumes); the protocol
fingerprint. All acquisitions must share **one** fingerprint: a calibration is for one protocol, and the fit
refuses data whose fingerprint differs from the calibration's.

In [ ]:
rows, bad = [], []
FP = {}
for i, f in ACQ.items():
    img, m, n = nib.load(f['dwi']), nib.load(f['mask']), nib.load(f['noise'])
    bv = np.loadtxt(f['bval']).ravel()
    bc = np.loadtxt(f['bvec']); bc = bc.T if bc.shape[0] == 3 and bc.shape[1] != 3 else bc
    nm = np.asarray(n.dataobj, np.float64); nm = nm[..., 0] if nm.ndim == 4 else nm
    mk = np.asarray(m.dataobj) > 0
    sh = img.shape
    if sh[3] != bv.size or bc.shape[0] != bv.size:
        bad.append(f'{i}: {sh[3]} volumes, {bv.size} b-values, {bc.shape[0]} b-vectors')
    if m.shape[:3] != sh[:3] or nm.shape != sh[:3]:
        bad.append(f'{i}: grids differ (DWI {sh[:3]}, mask {m.shape[:3]}, noise {nm.shape})'); continue
    if not (np.allclose(m.affine, img.affine, atol=1e-3) and np.allclose(n.affine, img.affine, atol=1e-3)):
        bad.append(f'{i}: mask or noise map with a different orientation from the DWI')
    v = nm[mk]; ok = np.isfinite(v) & (v > 0)
    if ok.mean() < 0.5:
        bad.append(f'{i}: only {ok.mean():.0%} of the mask has a valid noise value')
    b0 = np.where(bv < 100)[0]
    if b0.size == 0:
        bad.append(f'{i}: no b=0 volume (b < 100)'); continue
    s0 = np.mean([np.asarray(img.dataobj[..., k], np.float64)[mk] for k in b0], axis=0)
    FP[i] = protocol_fingerprint(bv)
    rows.append(dict(id=i, volumes=sh[3], b0=int(b0.size),
                     shells=' '.join(f'{int(b)}x{c}' for b, c in FP[i]['shells']),
                     mask_voxels=int(mk.sum()), noise_valid_pct=round(100 * ok.mean(), 1),
                     sigma_median=round(float(np.median(v[ok])), 2) if ok.any() else np.nan,
                     snr_thermal=round(float(np.median(s0[ok] / v[ok])), 1) if ok.any() else np.nan))
CHECK_TABLE = pd.DataFrame(rows).set_index('id')
display(CHECK_TABLE)
# one protocol: group the acquisitions by fingerprint
groups = {}
for i, fp in FP.items():
    key = next((k for k in groups if not fingerprint_mismatches(FP[k], fp)), i)
    groups.setdefault(key, []).append(i)
if len(groups) > 1:
    for k, ids in groups.items():
        print(f'  fingerprint of {k}: {FP[k]["n_volumes"]} volumes, shells {FP[k]["shells"]} -> {len(ids)} acquisitions')
    bad.append(f'{len(groups)} different protocols in the sample: keep one (ACQUISITION_DIRS) and rerun')
for p in bad:
    print('  [!]', p)
assert not bad, 'fix the inputs above before calibrating'
print(f'OK: {len(FP)} acquisitions, one protocol')

## 3. Calibrate each acquisition

One record per acquisition in `OUTPUT_DIR/records/` (complete GCV curves, chosen λ, noise, fingerprint).
Already-calibrated acquisitions are reloaded, not recalibrated (`FORCE = True` to redo them): an
interrupted run resumes here.

In [ ]:
os.makedirs(f'{OUTPUT_DIR}/records', exist_ok=True)
RECORDS, IDS = [], []
t_all = time.time()
for k, (i, f) in enumerate(ACQ.items(), 1):
    rec_path = f"{OUTPUT_DIR}/records/{i.replace(os.sep, '__')}.json"
    if os.path.isfile(rec_path) and not FORCE:
        rec = json.load(open(rec_path))
        print(f'[{k}/{len(ACQ)}] {i}: already calibrated -> loaded')
    else:
        data, affine, bvals, bvecs, mask = load_data(f['dwi'], f['bval'], f['bvec'], mask_path=f['mask'], verbose=False)
        nimg = nib.load(f['noise']); noise = np.asarray(nimg.dataobj, np.float64)
        noise = noise[..., 0] if noise.ndim == 4 else noise
        t0 = time.time()
        rec = DBSI_Adaptive().calibrate(np.asarray(data, np.float32), bvals, bvecs, np.asarray(mask, bool),
                                        n_calibration_voxels=N_CALIBRATION_VOXELS, noise_map=noise)
        json.dump(rec, open(rec_path, 'w'), indent=1)
        h = rec['hyperparameters']
        print(f"[{k}/{len(ACQ)}] {i}: lambda_aniso {h['lambda_aniso']:.4g}, lambda_iso {h['lambda_iso']:.4g}, "
              f"n_iso {h['n_iso']}  ({time.time() - t0:.0f} s)")
    RECORDS.append(rec); IDS.append(i)
print(f'{len(RECORDS)} records ({(time.time() - t_all) / 60:.1f} min)')
display(pd.DataFrame([dict(id=i, snr=r['noise']['snr'], lambda_aniso=r['hyperparameters']['lambda_aniso'],
                           lambda_iso=r['hyperparameters']['lambda_iso'], n_iso=r['hyperparameters']['n_iso'],
                           lambda_iso_capped=r['selection']['lambda_iso_capped'],
                           toolbox=r.get('toolbox_version')) for i, r in zip(IDS, RECORDS)]).set_index('id'))

## 4. One calibration for the protocol

`stability` says how often the choice survives resampling the acquisitions (bootstrap); `cost` how far each
acquisition is from its own optimum, in grid steps. A $\lambda$ on the edge of its grid, or a $\lambda_\text{iso}$
lowered by the cap, is flagged: both deserve a look before the numbers are frozen.

In [ ]:
CAL = calibrate_protocol(RECORDS, name=PROTOCOL_NAME, ids=IDS, rule=RULE, lambda_iso_cap_rule=LAMBDA_ISO_CAP_RULE)
JSON_PATH = f'{OUTPUT_DIR}/{PROTOCOL_NAME}_calibration.json'
SHA = save_protocol_calibration(CAL, JSON_PATH)
H, AG = CAL['hyperparameters'], CAL['aggregation']
print(f"protocol {PROTOCOL_NAME}: {AG['n_acquisitions']} acquisitions, rule {AG['rule']}")
print(f"  lambda_aniso = {H['lambda_aniso']!r}   (grid edge: {AG['lambda_aniso_at_grid_edge']})")
print(f"  lambda_iso   = {H['lambda_iso']!r}   (before the cap {AG['lambda_iso_before_cap']:.6g}, capped: "
      f"{AG['lambda_iso_capped']}, grid edge: {AG['lambda_iso_at_grid_edge']})")
print(f"  n_iso        = {H['n_iso']}")
if AG.get('alternative'):
    a = AG['alternative']
    print(f"  other rule ({a['rule']}): lambda_aniso {a['lambda_aniso']:.6g}, lambda_iso {a['lambda_iso']:.6g}")
display(pd.Series(CAL['stability']).to_frame('stability'))
display(pd.Series(CAL['cost']).to_frame('cost'))
print(f'\nwritten: {JSON_PATH}\nsha256:  {SHA}')

## 5. The curves behind the choice

Left and centre: each acquisition's GCV curve (divided by its minimum, so the shapes compare), its own
minimum (dot) and the protocol choice (dashed line). Right: each acquisition's own $\lambda_\text{aniso}$ against its
SNR — the dependence that a per-acquisition calibration would carry into a group comparison.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for r in RECORDS:
    for j, key in enumerate(('lambda_aniso', 'lambda_iso')):
        c = r['curves'][key]
        if c.get('gcv') is None or c.get('lambda_grid') is None:
            continue
        g, v = np.asarray(c['lambda_grid'], float), np.asarray(c['gcv'], float)
        okv = np.isfinite(v) & (v > 0)
        if not okv.any():
            continue
        ax[j].plot(g[okv], v[okv] / v[okv].min(), color='0.55', lw=1)
        k = np.argmin(np.where(okv, v, np.inf)); ax[j].plot(g[k], 1.0, 'o', color='C0', ms=4)
for j, key in enumerate(('lambda_aniso', 'lambda_iso')):
    ax[j].axvline(H[key], color='C3', ls='--', lw=1.5, label=f'protocol: {H[key]:.4g}')
    ax[j].set_xscale('log'); ax[j].set_yscale('log'); ax[j].set_xlabel(key); ax[j].set_ylabel('GCV / its minimum')
    ax[j].legend(fontsize=8)
snr = [r['noise']['snr'] for r in RECORDS]; la = [r['hyperparameters']['lambda_aniso'] for r in RECORDS]
ax[2].scatter(snr, la, color='C0'); ax[2].axhline(H['lambda_aniso'], color='C3', ls='--', lw=1.5)
ax[2].set_yscale('log'); ax[2].set_xlabel('thermal SNR'); ax[2].set_ylabel("acquisition's own lambda_aniso")
fig.suptitle(f'Protocol calibration {PROTOCOL_NAME} ({len(RECORDS)} acquisitions)')
fig.tight_layout(); fig.savefig(f'{OUTPUT_DIR}/{PROTOCOL_NAME}_calibration_curves.png', dpi=130); plt.show()

## 6. The command line

**Recommended: the JSON.** `--protocol-calibration` imposes the λ, `n_iso` and the dictionary options the λ were
calibrated on, **refuses** an acquisition with a different protocol fingerprint, and writes the file's sha256 in
every run report.

**Alternative: the numbers.** `--lambda-aniso`, `--lambda-iso`, `--n-iso` and `--n-dirs` give the same fit on an
acquisition of this protocol, but nothing checks that the acquisition *is* of this protocol, and the run report
does not say where the numbers came from. Use them where the JSON cannot travel.

In [ ]:
import shlex
f = next(iter(ACQ.values()))           # an example acquisition: replace its paths with the one to fit
cli = 'python scripts/run_dbsi.py'
base = (f"{cli} \\\n    --dwi {shlex.quote(f['dwi'])} \\\n    --bval {shlex.quote(f['bval'])} \\\n"
        f"    --bvec {shlex.quote(f['bvec'])} \\\n    --mask {shlex.quote(f['mask'])} \\\n"
        f"    --noise {shlex.quote(f['noise'])} \\\n    --out <output_dir>")
print('# recommended: the protocol calibration file')
print(base + f" \\\n    --protocol-calibration {shlex.quote(JSON_PATH)}\n")
print('# alternative: the numbers (no fingerprint check)')
print(base + f" \\\n    --skip-calibration --lambda-aniso {H['lambda_aniso']!r} --lambda-iso {H['lambda_iso']!r}"
             f" --n-iso {H['n_iso']} --n-dirs {CAL['model']['n_dirs']}\n")
print('# in Python')
print(f"model = DBSI_Adaptive.from_calibration({JSON_PATH!r})")